# BirdCLEF 2026 — CNN + Transformer Hybrid (Pipeline 02)
This notebook implements the high-end CNN→Transformer hybrid pipeline.

Architecture: `Mel Spectrogram → CNN Feature Extractor → Spatial Tokens → Transformer Encoder → Sigmoid Head`

**Why this matters:** Bird calls have strong temporal structure and overlapping events that attention mechanisms handle better than CNNs alone.

## 1. Setup & Imports

In [1]:
import os
import gc
import sys
import math
import time
import glob
import random
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import librosa
import soundfile as sf
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torchaudio
import torchaudio.transforms as T

import timm
import albumentations as A
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


## 2. Configuration & Paths

In [2]:
class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    # CNN backbone pretrained weights (timm model hub path on Kaggle)
    MODEL_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth')

    # Audio
    SR = 32000
    WINDOW_SECONDS = 5
    HOP_SECONDS = 2.5

    # Mel Spectrogram
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000

    # Transformer
    D_MODEL = 256       # token embedding dim
    NHEAD = 8           # attention heads
    NUM_LAYERS = 4      # transformer encoder layers
    DIM_FF = 1024       # feed-forward dim inside transformer
    DROPOUT = 0.1

    # Training
    SEED = 42
    BATCH_SIZE = 16     # lower than P01 due to VRAM cost of transformer
    EPOCHS = 15
    LR = 5e-5
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0

    # Model
    CNN_BACKBONE = 'tf_efficientnet_b0'
    NUM_CLASSES = 0  # populated automatically

CFG = Config()

print('Loading labels...')
train_df = pd.read_csv(CFG.TRAIN_CSV)
ss_df    = pd.read_csv(CFG.SOUNDSCAPE_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))

train_labels = sorted(train_df['primary_label'].unique())
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
unique_labels = submission_labels
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

train_df['label_id'] = train_df['primary_label'].map(label_to_id)
CFG.NUM_CLASSES = len(unique_labels)

print(f'Clip labels: {len(train_labels)} | Submission classes: {CFG.NUM_CLASSES}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Loading labels...
Clip labels: 206 | Submission classes: 234
Device: cuda


## 3. Utilities

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

## 4. Dataset & Data Processing

In [4]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        try:
            info = sf.info(audio_path)
            total_samples = info.frames
            if total_samples > self.window_samples:
                start = random.randint(0, total_samples - self.window_samples) if self.is_train else 0
                y, _ = sf.read(audio_path, start=start, frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(audio_path, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)

        if self.is_train and random.random() < 0.5:
            y = y + 0.005 * np.random.randn(len(y))

        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel = self.mel_transform(y_tensor)
        mel = self.amplitude_to_db(mel)
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        image = torch.stack([mel, mel, mel])  # (3, n_mels, T)

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0
        if 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            try:
                for sl in ast.literal_eval(row['secondary_labels']):
                    if sl in label_to_id:
                        target[label_to_id[sl]] = 1.0
            except: pass
        return image, target


In [5]:
class SoundscapeDataset(Dataset):
    def __init__(self, df, audio_dir):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        h, m, s = map(int, row['start'].split(':'))
        start_sample = (h * 3600 + m * 60 + s) * CFG.SR
        try:
            y, _ = sf.read(audio_path, start=start_sample,
                           stop=start_sample + self.window_samples, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)

        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel = self.mel_transform(y_tensor)
        mel = self.amplitude_to_db(mel)
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        image = torch.stack([mel, mel, mel])

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        for label in str(row['primary_label']).split(';'):
            if label in label_to_id:
                target[label_to_id[label]] = 1.0
        return image, target


## 5. Augmentations (SpecAugment + Mixup)

In [6]:
def mixup_data(x, y, alpha=0.4):
    """Mixup augmentation for spectrogram tensors."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class SpecAugment(nn.Module):
    """Frequency and time masking on spectrogram tensors (B,C,F,T)."""
    def __init__(self, freq_mask=16, time_mask=32, num_masks=2):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask
        self.num_masks = num_masks

    def forward(self, x):
        B, C, F, T = x.shape
        for _ in range(self.num_masks):
            # Frequency mask
            f = random.randint(0, self.freq_mask)
            f0 = random.randint(0, max(1, F - f))
            x[:, :, f0:f0+f, :] = 0
            # Time mask
            t = random.randint(0, self.time_mask)
            t0 = random.randint(0, max(1, T - t))
            x[:, :, :, t0:t0+t] = 0
        return x


## 6. Model Architecture — CNN + Transformer Hybrid

```
Mel Spectrogram (3, N_MELS, T)
      ↓
CNN Feature Extractor (EfficientNet-B0 features)
      ↓
Spatial Feature Map (B, C_feat, H', W') → reshape to tokens (B, H'*W', C_feat)
      ↓
Linear projection to D_MODEL + Positional Encoding
      ↓
Transformer Encoder (N layers, NHEAD heads)
      ↓
Global Average Pooling over token dim → (B, D_MODEL)
      ↓
Dropout → Linear → Multi-label Sigmoid Head
```

In [7]:
class PositionalEncoding(nn.Module):
    """Learnable positional embedding for sequence of tokens."""
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        seq_len = x.size(1)
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        return self.dropout(x + self.embedding(pos))


class CNNTransformerHybrid(nn.Module):
    """
    CNN frontend extracts spatial feature maps from the mel spectrogram.
    These maps are unrolled into a sequence of tokens and fed into a
    Transformer Encoder to model long-range temporal/spectral dependencies.
    """
    def __init__(self, cnn_backbone, num_classes, d_model, nhead,
                 num_layers, dim_ff, dropout, model_path=None):
        super().__init__()

        # ── CNN Frontend ──────────────────────────────────────────
        if model_path is not None and Path(model_path).exists():
            backbone = timm.create_model(
                cnn_backbone, pretrained=False,
                in_chans=3, features_only=True
            )
            state_dict = torch.load(model_path, map_location='cpu')
            if 'state_dict' in state_dict: state_dict = state_dict['state_dict']
            elif 'model' in state_dict: state_dict = state_dict['model']
            backbone.load_state_dict(state_dict, strict=False)
        else:
            backbone = timm.create_model(
                cnn_backbone, pretrained=False,
                in_chans=3, features_only=True
            )
        # Use the last feature stage only (richest spatial representation)
        self.cnn = backbone
        # Probe feature channel count with a dummy pass
        with torch.no_grad():
            dummy = torch.zeros(1, 3, CFG.N_MELS, CFG.N_MELS)
            feats = self.cnn(dummy)
            feat_channels = feats[-1].shape[1]  # last stage channels

        # ── Token Projection ──────────────────────────────────────
        self.token_proj = nn.Sequential(
            nn.Conv2d(feat_channels, d_model, kernel_size=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

        # ── Positional Encoding ───────────────────────────────────
        self.pos_enc = PositionalEncoding(d_model, max_len=4096, dropout=dropout)

        # ── Transformer Encoder ───────────────────────────────────
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # ── Classification Head ───────────────────────────────────
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        # x: (B, 3, N_MELS, T)
        feat_maps = self.cnn(x)      # list of feature maps; take last
        f = feat_maps[-1]            # (B, C, H', W')
        f = self.token_proj(f)       # (B, d_model, H', W')
        B, D, H, W = f.shape
        tokens = f.flatten(2).transpose(1, 2)  # (B, H*W, d_model)
        tokens = self.pos_enc(tokens)           # add position
        tokens = self.transformer(tokens)        # (B, H*W, d_model)
        pooled = tokens.mean(dim=1)             # global avg pool over tokens
        return self.head(pooled)                # (B, num_classes) — raw logits


## 7. Loss & Optimization

In [8]:
def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def get_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def get_criterion():
    # BCEWithLogitsLoss for multi-label; pos_weight omitted here but can be tuned
    return nn.BCEWithLogitsLoss()


## 8. Training & Validation Loops

In [9]:
spec_augment = SpecAugment(freq_mask=16, time_mask=32, num_masks=2)

def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    spec_augment.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc='Train', leave=False):
        images, targets = images.to(device), targets.to(device)

        # SpecAugment on batch
        images = spec_augment(images)

        # Mixup
        images, targets_a, targets_b, lam = mixup_data(images, targets, alpha=0.4)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)


In [10]:
def valid_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_targets = [], []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())

    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)

    # Competition metric: macro ROC-AUC skipping classes with no positives
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        if len(np.unique(true_targets[:, i])) > 1:
            auc_scores.append(roc_auc_score(true_targets[:, i], preds[:, i]))
    final_auc = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_auc


## 9. Main Execution

In [11]:
# ── Data Preparation ────────────────────────────────────────────
df = train_df.copy()
if 'label_id' not in df.columns:
    df['label_id'] = df['primary_label'].map(label_to_id)

# Duplicate rare classes so stratified split stays valid
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()
if rare_birds:
    print(f'Found {len(rare_birds)} species with 1 sample — duplicating for stratification.')
    df = pd.concat([df, df[df['label_id'].isin(rare_birds)]], ignore_index=True)

# 80/20 zero-leakage splits
train_df_clips, valid_df_clips = train_test_split(
    df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED
)
train_ss_df, valid_ss_df = train_test_split(
    ss_df, test_size=0.2, random_state=CFG.SEED
)

print(f"{'='*20} Zero-Leakage Split {'='*20}")
print(f'Train Clips: {len(train_df_clips)} | Valid Clips: {len(valid_df_clips)}')
print(f'Train SS   : {len(train_ss_df)} | Valid SS   : {len(valid_ss_df)}')


Found 4 species with 1 sample — duplicating for stratification.
==================== Zero-Leakage Split ====================
Train Clips: 28442 | Valid Clips: 7111
Train SS   : 1182 | Valid SS   : 296


In [12]:
# ── Weighted Sampler (class-balanced training) ───────────────────
def get_sampler(df_subset):
    class_counts = df_subset['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / (class_counts + 1e-6)
    weights = df_subset['label_id'].map(
        lambda x: class_weights[x] if x < len(class_weights) else 0
    ).values
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_sampler = get_sampler(train_df_clips)

# ── Datasets ────────────────────────────────────────────────────────
train_clip_ds      = BirdDataset(train_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=True)
train_soundscape_ds = SoundscapeDataset(train_ss_df, CFG.SOUNDSCAPE_DIR)
train_ds           = ConcatDataset([train_clip_ds, train_soundscape_ds])

valid_ds_clips = BirdDataset(valid_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=False)
valid_ds_ss    = SoundscapeDataset(valid_ss_df, CFG.SOUNDSCAPE_DIR)

# Eval-only (no augmentation) for training-set AUC measurement
train_clip_eval_ds = BirdDataset(train_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=False)
train_ss_eval_ds   = SoundscapeDataset(train_ss_df, CFG.SOUNDSCAPE_DIR)

# ── DataLoaders ─────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds, batch_size=CFG.BATCH_SIZE,
    sampler=train_sampler,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)
train_clip_eval_loader = DataLoader(
    train_clip_eval_ds, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
train_ss_eval_loader = DataLoader(
    train_ss_eval_ds, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
valid_loader_clips = DataLoader(
    valid_ds_clips, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
valid_loader_ss = DataLoader(
    valid_ds_ss, batch_size=CFG.BATCH_SIZE,
    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
print('All datasets and loaders ready.')


All datasets and loaders ready.


In [13]:
# ── Model + Training Setup ───────────────────────────────────────
model = CNNTransformerHybrid(
    cnn_backbone=CFG.CNN_BACKBONE,
    num_classes=CFG.NUM_CLASSES,
    d_model=CFG.D_MODEL,
    nhead=CFG.NHEAD,
    num_layers=CFG.NUM_LAYERS,
    dim_ff=CFG.DIM_FF,
    dropout=CFG.DROPOUT,
    model_path=CFG.MODEL_DIR
).to(device)

optimizer = get_optimizer(model)
scheduler = get_scheduler(optimizer)
criterion = get_criterion()
scaler    = torch.cuda.amp.GradScaler()

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Training Loop with 4-Way Pure Metrics ───────────────────────
best_score = 0.0
for epoch in range(1, CFG.EPOCHS + 1):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)

    # Train AUC — evaluated on training data (no augmentation, no leakage)
    _, train_auc_clips = valid_epoch(model, train_clip_eval_loader, criterion, device)
    _, train_auc_ss    = valid_epoch(model, train_ss_eval_loader,   criterion, device)

    # Val AUC — evaluated on unseen held-out 20%
    _, val_auc_clips = valid_epoch(model, valid_loader_clips, criterion, device)
    _, val_auc_ss    = valid_epoch(model, valid_loader_ss,    criterion, device)

    scheduler.step()

    # Overfit gaps (apples-to-apples)
    p_clip = abs(train_auc_clips - val_auc_clips)  # monitoring only
    p_ss   = abs(train_auc_ss   - val_auc_ss)      # used in checkpoint metric

    # Robust score: primary objective = val_ss; penalised by SS overfit gap
    robust_score = val_auc_ss - p_ss

    duration = time.time() - start_time
    print(f'Epoch {epoch:02d} | Loss: {train_loss:.4f} | Time: {int(duration)}s')
    print(f'  Clips -> Train: {train_auc_clips:.4f} | Val: {val_auc_clips:.4f} | Gap: {p_clip:.4f}  [monitoring]')
    print(f'  SS    -> Train: {train_auc_ss:.4f}   | Val: {val_auc_ss:.4f}   | Gap: {p_ss:.4f}  [checkpoint]')
    print(f'  Robust Score (val_ss - ss_gap): {robust_score:.4f}')

    if robust_score > best_score:
        best_score = robust_score
        torch.save(model.state_dict(), 'best_model_hybrid.pth')
        print(f'  !!! NEW BEST SAVED | Robust: {best_score:.4f} | Val SS AUC: {val_auc_ss:.4f} | SS Gap: {p_ss:.4f} !!!')


Model parameters: 7,945,574


Train:   0%|          | 0/1778 [00:00<?, ?it/s]

Valid:   0%|          | 0/1778 [00:00<?, ?it/s]

Valid:   0%|          | 0/74 [00:00<?, ?it/s]

Valid:   0%|          | 0/445 [00:00<?, ?it/s]

...

Valid:   0%|          | 0/1778 [00:00<?, ?it/s]

Valid:   0%|          | 0/74 [00:00<?, ?it/s]

Valid:   0%|          | 0/445 [00:00<?, ?it/s]

Valid:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 15 | Loss: 0.0161 | Time: 1179s
  Clips -> Train: 0.8856 | Val: 0.8657 | Gap: 0.0199  [monitoring]
  SS    -> Train: 0.5593   | Val: 0.6072   | Gap: 0.0479  [checkpoint]
  Robust Score (val_ss - ss_gap): 0.5593


## 10. Inference & Submission

In [14]:
# Load Best Model
print('Loading best model for inference...')
infer_model = CNNTransformerHybrid(
    cnn_backbone=CFG.CNN_BACKBONE,
    num_classes=CFG.NUM_CLASSES,
    d_model=CFG.D_MODEL, nhead=CFG.NHEAD,
    num_layers=CFG.NUM_LAYERS, dim_ff=CFG.DIM_FF,
    dropout=CFG.DROPOUT, model_path=None
).to(device)
try:
    infer_model.load_state_dict(torch.load('best_model_hybrid.pth', map_location=device))
    infer_model.eval()
    print('Model loaded successfully.')
except Exception as e:
    print(f'Warning: Could not load best_model_hybrid.pth. Error: {e}')


Loading best model for inference...
Model loaded successfully.


In [15]:
# Fallback Logic
# On Kaggle, test_soundscapes only exists in the hidden evaluation environment.
# During dry-runs/debug the folder is absent — use training soundscapes instead.
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = []
if os.path.exists(TEST_DIR):
    test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))

if len(test_files) == 0:
    print('⚠️  FALLBACK ACTIVE: No test files found. Using training soundscapes as dry-run.')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]  # first 5 for speed
    IS_DRY_RUN = True
else:
    print(f'✅ Found {len(test_files)} test files.')
    IS_DRY_RUN = False


⚠️  FALLBACK ACTIVE: No test files found. Using training soundscapes as dry-run.


In [16]:
# Sliding Window Inference (non-overlapping 5-s segments matching soundscape schema)
mel_transform  = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_predictions, all_row_ids = [], []

print(f"\n{'='*20} Starting Inference {'='*20}")
for audio_path in tqdm(test_files, desc='Processing Soundscapes'):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    try:
        y, _ = sf.read(audio_path, always_2d=True)
        y = y.mean(axis=1)  # mono
    except Exception as e:
        print(f'Error reading {audio_path}: {e}')
        continue

    y_tensor = torch.tensor(y, dtype=torch.float32).to(device)
    total_samples  = len(y_tensor)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    n_segments     = math.ceil(total_samples / window_samples)

    for seg_idx in range(n_segments):
        start_sample  = seg_idx * window_samples
        end_time_sec  = (seg_idx + 1) * CFG.WINDOW_SECONDS
        row_id        = f'{filename}_{end_time_sec}'

        segment = y_tensor[start_sample: start_sample + window_samples]
        if len(segment) < window_samples:
            segment = F.pad(segment, (0, window_samples - len(segment)))

        with torch.no_grad():
            mel = mel_transform(segment)
            mel = amplitude_to_db(mel)
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
            image = torch.stack([mel, mel, mel]).unsqueeze(0)  # (1,3,F,T)
            logits = infer_model(image)
            probs  = torch.sigmoid(logits).squeeze(0).cpu().numpy()

        all_row_ids.append(row_id)
        all_predictions.append(probs)



==================== Starting Inference ====================


Processing Soundscapes:   0%|          | 0/5 [00:00<?, ?it/s]

In [17]:
# Submission Formatting
print('\nFormatting submission...')
prediction_df = pd.DataFrame(all_predictions, columns=unique_labels)
submission_df = prediction_df.reindex(columns=submission_labels, fill_value=0.0)
submission_df.insert(0, 'row_id', all_row_ids)
submission_df = submission_df[['row_id'] + submission_labels]

# Sanity checks
expected_cols = len(submission_labels) + 1
assert submission_df.shape[1] == expected_cols, \
    f'Column mismatch: {submission_df.shape[1]} vs {expected_cols}'
assert len(submission_df) == len(all_row_ids), \
    f'Row mismatch: {len(submission_df)} vs {len(all_row_ids)}'
assert not submission_df.isnull().values.any(), 'Submission contains NaNs'

submission_df.to_csv('submission.csv', index=False)
print(f'✅ Submission saved → submission.csv  shape={submission_df.shape}')
display(submission_df.head(3))



Formatting submission...
✅ Submission saved → submission.csv  shape=(60, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.002334,0.002849,0.002727,0.002123,0.005131,0.002670,0.002299,0.011736,0.003264,...,0.001766,0.012990,0.001855,0.002422,0.001697,0.002086,0.002896,0.006423,0.002771,0.002342
1,BC2026_Train_0001_S08_20250606_030007_10,0.002341,0.002857,0.002756,0.002147,0.005087,0.002660,0.002287,0.011815,0.003329,...,0.001775,0.012798,0.001878,0.002470,0.001698,0.002107,0.002974,0.006483,0.002805,0.002391
2,BC2026_Train_0001_S08_20250606_030007_15,0.002316,0.002886,0.002720,0.002112,0.005085,0.002702,0.002297,0.011771,0.003320,...,0.001756,0.012918,0.001879,0.002437,0.001676,0.002085,0.002895,0.006349,0.002764,0.002367
